### DECISION TREE PRUNING

In [ ]:
import sklearn
sklearn.__version__

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.tree import plot_tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier


In [ ]:
data = load_breast_cancer()
print(data.DESCR)

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

In [ ]:
X_train.shape

In [ ]:
unpruned_tree = DecisionTreeClassifier(random_state=0)
unpruned_tree.fit(X_train,y_train)

In [ ]:
pred=unpruned_tree.predict(X_test)
from sklearn.metrics import accuracy_score
print("Test Accuracy ->",accuracy_score(y_test, pred))

In [ ]:
print("Train Accuracy ->",unpruned_tree.score(X_train, y_train))

In [ ]:
from sklearn import tree
plt.figure(figsize=(15,10))
tree.plot_tree(unpruned_tree,filled=True)
plt.title("Unpruned Decision Tree")

# Pre Pruning



The class:`DecisionTreeClassifier` takes parameters such as
``min_samples_leaf`` and ``max_depth`` to prevent a tree from overfitting.

In [ ]:
print(unpruned_tree.get_params())

In [ ]:

pre_pruned_tree = DecisionTreeClassifier(max_leaf_nodes=4,random_state=0)
pre_pruned_tree.fit(X_train,y_train)

In [ ]:
print("Train Accuracy ->",pre_pruned_tree.score(X_train, y_train))

In [ ]:
pred=pre_pruned_tree.predict(X_test)
print("Test accuracy ->",accuracy_score(y_test, pred))

In [ ]:
plt.figure(figsize=(15,8))
tree.plot_tree(pre_pruned_tree,filled=True)
plt.title("Pre-pruned Decision Tree")


# Post pruning





Cost complexity pruning is used to control the size of a tree. In
:class:`DecisionTreeClassifier`, this pruning technique is parameterized by the
cost complexity parameter, ``ccp_alpha``. Greater values of ``ccp_alpha``
increase the number of nodes pruned. Here we show the effect of
``ccp_alpha`` on regularizing the trees and how to choose a ``ccp_alpha``
based on validation scores.

In [ ]:
path = unpruned_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

In [ ]:
path

In [ ]:
ccp_alphas

In [ ]:
impurities

In [ ]:
clfs = []
for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=0, ccp_alpha=ccp_alpha)
    clf.fit(X_train, y_train)
    clfs.append(clf)
print("Number of nodes in the last tree is: {} with ccp_alpha: {}".format(
      clfs[-1].tree_.node_count, ccp_alphas[-1]))

In [ ]:
import io
import imageio
import matplotlib.pyplot as plt
from sklearn import tree

images = []
for i, clf in enumerate(clfs):
    buf = io.BytesIO()
    plt.figure(figsize=(10, 8))
    tree.plot_tree(clf, filled=True, fontsize=6)
    plt.title(f"Tree with ccp_alpha={ccp_alphas[i]:.4f}")
    plt.savefig(buf, format='png')
    plt.close()

    images.append(imageio.imread(buf))

imageio.mimsave('pruned_trees_in_memory.gif', images, duration=1000) # Increased duration to 10 seconds
from IPython.display import Image
Image(filename='pruned_trees_in_memory.gif',embed=True)

# Total Impurity  Vs Alpha

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.plot(ccp_alphas[:-1], impurities[:-1], marker="o", drawstyle="steps-post")
ax.set_xlabel("effective alpha")
ax.set_ylabel("total impurity of leaves")
ax.set_title("Total Impurity vs effective alpha for training set")

For this plot, we remove the last element in
``clfs`` and ``ccp_alphas``, because it is the trivial tree with only one
node. .



# Number of nodes Vs Alpha

In [ ]:
clfs = clfs[:-1]
ccp_alphas = ccp_alphas[:-1]

node_counts = [clf.tree_.node_count for clf in clfs]
depth = [clf.tree_.max_depth for clf in clfs]
fig, ax = plt.subplots(2, 1,figsize=(15, 8))
ax[0].plot(ccp_alphas, node_counts, marker="o", drawstyle="steps-post")
ax[0].set_xlabel("Alpha")
ax[0].set_ylabel("Number of nodes")
ax[0].set_title("Number of nodes vs Alpha")
ax[1].plot(ccp_alphas, depth, marker="o", drawstyle="steps-post")
ax[1].set_xlabel("Alpha")
ax[1].set_ylabel("Depth of tree")
ax[1].set_title("Depth vs Alpha")
fig.tight_layout()



# Accuracy Vs Alpha

In [ ]:
train_scores = [clf.score(X_train, y_train) for clf in clfs]
test_scores = [clf.score(X_test, y_test) for clf in clfs]

fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlabel("alpha")
ax.set_ylabel("accuracy")
ax.set_title("Accuracy vs alpha for training and testing sets")
ax.plot(ccp_alphas, train_scores, marker='o', label="Train",
        drawstyle="steps-post")
ax.plot(ccp_alphas, test_scores, marker='o', label="Test",
        drawstyle="steps-post")

optimal_alpha_index = test_scores.index(max(test_scores))
optimal_alpha = ccp_alphas[optimal_alpha_index]
highest_test_score = test_scores[optimal_alpha_index]


ax.plot(optimal_alpha, highest_test_score, 'o', color='red', markersize=15, fillstyle='none', label=f'Highest Test Accuracy (alpha={optimal_alpha:.4f})')

ax.hlines(highest_test_score, ax.get_xlim()[0], optimal_alpha, color='gray', linestyle='--', alpha=0.7)
ax.vlines(optimal_alpha, ax.get_ylim()[0], highest_test_score, color='gray', linestyle='--', alpha=0.7)


ax.legend()
plt.show()

In [ ]:
post_pruned_tree = DecisionTreeClassifier(random_state=0, ccp_alpha=0.0114)
post_pruned_tree.fit(X_train,y_train)

In [ ]:
pred=post_pruned_tree.predict(X_test)
from sklearn.metrics import accuracy_score
print("Train Accuracy ->",post_pruned_tree.score(X_train, y_train))
print("Test Accuracy ->",accuracy_score(y_test, pred))

In [ ]:
from sklearn import tree
plt.figure(figsize=(15,10))

tree.plot_tree(post_pruned_tree,filled=True)
plt.title("Post-Pruned Decision Tree")